# Day 2 | Lab 2.1: Crafting Precise Prompts with ChatPromptTemplate

**Duration:** ~1.5 hours

**Scenario.** Retail-banking customer support — preserved from the GM source notebook (banking scenarios already present). The lab demonstrates the difference between sloppy prompts and precisely-crafted ones using `ChatPromptTemplate`.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Build prompts using LangChain's `ChatPromptTemplate` from cell 1 — no raw f-string prompts.
2. Provide clear context, role, and constraints to control model output.
3. Apply few-shot examples to guide tone, format, and decision boundaries.
4. Read a poorly-crafted prompt's output, diagnose the issue, and rewrite the prompt.
5. Set up a reusable `prompt | llm` chain that can be composed with parsers (Lab 2.4) and memory (Module 3).

**Tools.** LangChain v1 · `langchain-openai` · `gpt-4.1-mini` (non-reasoning, supports temperature).

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' python-dotenv


## 2. API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


In [7]:
import textwrap

def wrap_print(text, width=100):
    """Prints text wrapped to a specified width."""
    print(textwrap.fill(text, width=width))
    print()

# Example usage with a long string
long_text = "This is a very long string that would normally require horizontal scrolling to read completely if it were just printed out directly to the console without any formatting. But with this helper function, it will be nicely wrapped!"

wrap_print(long_text)

This is a very long string that would normally require horizontal scrolling to read completely if it
were just printed out directly to the console without any formatting. But with this helper function,
it will be nicely wrapped!



In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# # Load environment variables from .env file
# load_dotenv()

# Initialize the model
# We use gpt-4o-mini for its efficiency and capability
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

# Helper function to print results neatly
def generate_and_print(prompt_text, scenario_title):
    print(f"--- {scenario_title} ---")
    print(f"Executing Prompt:\n\n'{prompt_text}'\n")

    prompt = ChatPromptTemplate.from_template(prompt_text)
    chain = prompt | llm
    response = chain.invoke({})

    print("\n\nLLM Response \n")
    wrap_print(response.content)
    print("-" * 30 + "\n")

-----

## **Scenario 1: Summarizing a Customer Loan Inquiry**

A loan officer needs a quick summary of a long customer email to assess eligibility at a glance.

### **Bad Prompt Example 👎**

This prompt is too vague. It doesn't specify the output format or the key information to extract.

**Code:**

In [9]:
customer_email = """
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.
"""

bad_prompt_1 = f"Summarize this email: {customer_email}"

generate_and_print(bad_prompt_1, "Scenario 1: Bad Prompt")

--- Scenario 1: Bad Prompt ---
Executing Prompt:

'Summarize this email: 
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.
'

LLM Response:  Priya Sharma, a loyal customer of over 10 years and a software developer earning
₹18,00,000 annually, is inquiring about obtaining a ₹5,00,000 personal loan for a family wedding.
She has a good credit score of around 780, no existing loans with the bank, and a credit card with a
₹2,00,000 limit. She requests information on the next steps.

------------------------------



*Critique:* The summary is okay, but it's unstructured and mixes all the information. A loan officer would have to read through the paragraph to find specific data points.

### **Good Prompt Example 👍**

This prompt is precise. It assigns a **role** (Loan Officer Assistant), provides clear **context**, specifies the **task**, and sets **constraints** on the output format (JSON).

**Code:**

In [14]:
customer_email = """
Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.
"""

good_prompt_1 = f"""
You are a highly efficient AI assistant for a bank's loan officer.
Your task is to extract key information from a customer's loan inquiry email and present it in a structured JSON format.

**Email Content:**
---
{customer_email}
---

**Instructions:**
1. Extract the following fields: customer_name, loan_amount, loan_purpose, annual_salary, cibil_score, and existing_liabilities.
2. If a specific piece of information is not mentioned, use "N/A".
3. The output must be a valid JSON object. Do not add any introductory text or explanations.
"""

generate_and_print(good_prompt_1, "Scenario 1: Good Prompt")

--- Scenario 1: Good Prompt ---
Executing Prompt:

'
You are a highly efficient AI assistant for a bank's loan officer. 
Your task is to extract key information from a customer's loan inquiry email and present it in a structured JSON format.

**Email Content:**
---

Hi, my name is Priya Sharma. I've been a loyal customer with your bank for over 10 years.
I'm writing to inquire about a personal loan. I'm a software developer at a tech firm, earning
an annual salary of ₹18,00,000. I need to borrow ₹5,00,000 for a family wedding. I have a
good credit history with a CIBIL score of around 780. I don't have any existing loans with
your bank but I do have a credit card with a ₹2,00,000 limit. Could you please let me know
the next steps? Thanks, Priya.

---

**Instructions:**
1. Extract the following fields: customer_name, loan_amount, loan_purpose, annual_salary, cibil_score, and existing_liabilities.
2. If a specific piece of information is not mentioned, use "N/A".
3. The output must be a v

*Critique:* This response is perfect. The JSON format is machine-readable and easy for the loan officer to quickly parse. All key information is clearly labeled.

-----

## **Scenario 2: Drafting a Customer Communication for a Failed Transaction**

A customer's online payment failed. The bank needs to send a clear, empathetic, and helpful notification.

### **Bad Prompt Example 👎**

This prompt lacks tone guidance and doesn't instruct the model on what helpful actions to suggest.

**Code:**

In [15]:
transaction_details = {
    "customer_name": "Mr. Verma",
    "amount": "₹3,500",
    "merchant": "Flipkart",
    "reason": "Incorrect CVV"
}

bad_prompt_2 = f"Tell {transaction_details['customer_name']} that their payment of {transaction_details['amount']} to {transaction_details['merchant']} failed because of {transaction_details['reason']}."

generate_and_print(bad_prompt_2, "Scenario 2: Bad Prompt")

--- Scenario 2: Bad Prompt ---
Executing Prompt:

'Tell Mr. Verma that their payment of ₹3,500 to Flipkart failed because of Incorrect CVV.'



LLM Response 

Sure! Here's a message you can send to Mr. Verma:  ---  Dear Mr. Verma,  We would like to inform you
that your payment of ₹3,500 to Flipkart has failed due to an incorrect CVV entered. Please verify
your card details and try again.  If you need any assistance, feel free to contact us.  Thank you.
---  Would you like me to make it more formal or casual?

------------------------------




*Critique:* This response is blunt, robotic, and unhelpful. It could cause customer anxiety and doesn't offer a solution.

### **Good Prompt Example 👍**

This prompt uses the **role-playing** technique, specifies the desired **tone** (empathetic and professional), and provides a clear **structure** with actionable next steps. This is a simple form of **few-shot prompting** by providing an example of the desired output style within the instructions.

**Code:**

In [16]:
transaction_details = {
    "customer_name": "Mr. Verma",
    "amount": "₹3,500",
    "merchant": "Flipkart",
    "reason": "Incorrect CVV"
}

good_prompt_2 = f"""
You are a Customer Service Bot for SecureBank.
Your task is to draft an SMS notification to a customer about a failed transaction.

**Tone:** Empathetic, professional, and reassuring.

**Instructions:**
1. Address the customer by name.
2. Clearly state that the transaction could not be processed.
3. Mention the amount and merchant.
4. Briefly and simply state the reason for the failure.
5. Provide a clear, immediate action the customer can take (e.g., "please try again, ensuring the details are correct").
6. End on a helpful note, providing a support channel.

**Customer and Transaction Details:**
- Customer Name: {transaction_details['customer_name']}
- Amount: {transaction_details['amount']}
- Merchant: {transaction_details['merchant']}
- Reason: {transaction_details['reason']}

Draft the SMS message.
"""

generate_and_print(good_prompt_2, "Scenario 2: Good Prompt")

--- Scenario 2: Good Prompt ---
Executing Prompt:

'
You are a Customer Service Bot for SecureBank. 
Your task is to draft an SMS notification to a customer about a failed transaction.

**Tone:** Empathetic, professional, and reassuring.

**Instructions:**
1. Address the customer by name.
2. Clearly state that the transaction could not be processed.
3. Mention the amount and merchant.
4. Briefly and simply state the reason for the failure.
5. Provide a clear, immediate action the customer can take (e.g., "please try again, ensuring the details are correct").
6. End on a helpful note, providing a support channel.

**Customer and Transaction Details:**
- Customer Name: Mr. Verma
- Amount: ₹3,500
- Merchant: Flipkart
- Reason: Incorrect CVV

Draft the SMS message.
'



LLM Response 

Dear Mr. Verma, your transaction of ₹3,500 at Flipkart could not be processed due to an incorrect
CVV. Please try again, ensuring the CVV entered is correct. If you need assistance, contact
SecureBank support

*Critique:* This response is excellent. It's polite, explains the problem clearly, offers an immediate solution, and provides a way to get more help, building customer trust.

-----

## **Challenge Section: Your Turn\!**

Now, apply what you've learned. For the following scenario, write a precise "good" prompt yourself.

**Scenario:** A bank wants to generate a marketing email to promote its new "GlobalTravel" credit card to existing customers who are identified as frequent travelers.

**Key Features of the Card:**

  * Zero foreign transaction fees.
  * Complimentary airport lounge access (4 per year).
  * 10,000 bonus reward points on signup.
  * Annual fee: ₹2,500.

**Your Task:**

1.  Create a variable named `challenge_prompt`.
2.  In this variable, write a prompt that instructs the LLM to draft a persuasive and personalized marketing email.
3.  **Your prompt should specify:**
      * The **role** of the AI (e.g., a marketing assistant).
      * The **target audience** (existing customers who travel frequently).
      * The desired **tone** (exciting, exclusive, benefits-focused).
      * A clear **structure** for the email (e.g., personalized greeting, hook, key features as bullet points, clear call-to-action).
      * The customer's name, which should be a placeholder like `[Customer Name]`.

**Write your code below:**

In [ ]:
#
# Your code here!
#
# challenge_prompt = """ ... your prompt ... """
# generate_and_print(challenge_prompt, "Challenge: Marketing Email")
#

-----

### **Key Takeaways**

1.  **Be Specific:** Vague prompts lead to vague answers. Tell the model exactly what you want.
2.  **Assign a Role:** Telling the LLM to act as a "Loan Officer" or "Marketing Expert" primes it for better, context-aware responses.
3.  **Define the Format:** Don't leave the output structure to chance. Explicitly ask for JSON, a list, an email, or a table.
4.  **Set the Tone:** The same information can be delivered in many ways. Specify the tone (e.g., formal, empathetic, persuasive) to match the use case.
5.  **Provide Context:** The LLM doesn't know what you know. Give it all the necessary background information to complete the task successfully.

---
## 5. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Role + context + constraints** | The 3 dimensions every prompt should explicitly set |
| **ChatPromptTemplate** | Replaces raw f-strings — composable, parameterizable, testable |
| **Few-shot examples** | Force tone, format, and decision boundaries the model can't infer from instructions alone |
| **Critique-then-rewrite** | Read every output critically; the second prompt is always better than the first |
| **`prompt | llm` chains** | Foundation for every chain in Modules 3, 4, 5 |

**Next Lab:** Lab 2.2 — Foundational Prompt Patterns (persona, N-shot, directional, template) 🎯


## 6. Stretch Exercise (Optional)

1. Take one of your existing banking prompts (or write one from scratch) and craft 3 versions: minimal, medium, fully-specified. Compare outputs side-by-side.
2. Add a `MessagesPlaceholder` slot for chat history to your `ChatPromptTemplate` — preview of Module 3.
3. Convert one of the prompts in this lab to use a Pydantic schema via `with_structured_output(...)`. Preview of Lab 2.4.
4. Build a 5-question evaluation set (input + ideal output) for the customer-email summary task. Run all 4 prompts in this lab against it and rank.
